In [1]:
from pathlib import Path

import numpy as np
import ovito
from IPython.display import display
from ipywidgets import Layout
from ovito.gui import create_ipywidget
from ovito.io import import_file
from ovito.vis import Viewport


# File di input con colonne: x y z
txt_path = Path("/Users/francesco/Desktop/computational-course/data/raw/fcc100a108.txt")

if not txt_path.exists():
    raise FileNotFoundError(f"File non trovato: {txt_path}")

# Legge le coordinate
coords = np.loadtxt(txt_path)

# Gestione del caso con una sola riga
if coords.ndim == 1:
    coords = coords.reshape(1, -1)

if coords.shape[1] < 3:
    raise ValueError(
        f"Il file deve contenere almeno 3 colonne x y z; "
        f"trovate {coords.shape[1]} colonne."
    )

coords = coords[:, :3]

print(f"Atomi letti: {len(coords)}")
print("Prime coordinate:")
print(coords[:5])


# ---------------------------------------------------------
# Conversione temporanea in XYZ, facilmente leggibile da OVITO
# ---------------------------------------------------------

OUT = Path("generated")
OUT.mkdir(exist_ok=True)

xyz_path = OUT / "fcc100a108.xyz"

with xyz_path.open("w", encoding="utf-8") as f:
    f.write(f"{len(coords)}\n")
    f.write("Ag FCC(100) imported from fcc100a108.txt\n")

    for x, y, z in coords:
        f.write(f"Ag {x:.8f} {y:.8f} {z:.8f}\n")


# ---------------------------------------------------------
# Visualizzazione con OVITO
# ---------------------------------------------------------

# Pulisce eventuali pipeline già presenti nella scena
for old_pipeline in list(ovito.scene.pipelines):
    old_pipeline.remove_from_scene()

pipeline = import_file(str(xyz_path))
pipeline.add_to_scene()

# Vista dall'alto: utile per una superficie FCC(100)
vp = Viewport(
    type=Viewport.Type.Ortho,
    camera_dir=(0.0, 0.0, -1.0),
)

vp.zoom_all()

ovito_view = create_ipywidget(
    vp,
    antialiasing=True,
    picking=False,
    layout=Layout(
        width="100%",
        height="520px",
    ),
)

display(ovito_view)

Atomi letti: 108
Prime coordinate:
[[2.0802 2.0802 0.    ]
 [0.     0.     0.    ]
 [0.     4.1604 0.    ]
 [2.0802 6.2406 0.    ]
 [0.     8.3208 0.    ]]


JupyterViewportWidget(layout=Layout(height='520px', width='100%'))

Ho importato il primo file che dovrebbe essere argento. ora voglio  visualizzare meglio il widget e poi procedo con il calcolo della distanza dei NN.

In [2]:
from IPython.display import display
from ipywidgets import GridBox, Layout, VBox, HTML

from ovito.gui import create_ipywidget
from ovito.vis import Viewport


# ---------------------------------------------------------
# Stile grafico per ciascun pannello
# ---------------------------------------------------------


def make_panel(
    view_type,
    title,
    plane,
    axis_html,
    camera_dir=None,
):
    kwargs = {"type": view_type}

    if camera_dir is not None:
        kwargs["camera_dir"] = camera_dir

    vp = Viewport(**kwargs)
    vp.zoom_all()

    viewer = create_ipywidget(
        vp,
        antialiasing=True,
        picking=False,
        layout=Layout(
            width="100%",
            height="310px",
        ),
    )

    # Barra superiore integrata
    header = HTML(
        value=f"""
        <div style="
            background:#202225;
            color:#f2f2f2;
            padding:7px 10px;
            font-family:-apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
            font-size:14px;
            border-bottom:1px solid #444;
            display:flex;
            align-items:center;
            justify-content:space-between;
        ">
            <span style="font-weight:600;">
                {title}
            </span>

            <span style="
                color:#aaa;
                font-size:12px;
                font-weight:500;
            ">
                {plane}
            </span>
        </div>
        """
    )

    # Barra inferiore con sistema cartesiano
    footer = HTML(
        value=f"""
        <div style="
            background:#202225;
            color:#ddd;
            padding:6px 10px;
            font-family:Menlo, Monaco, monospace;
            font-size:12px;
            border-top:1px solid #444;
            display:flex;
            align-items:center;
            justify-content:space-between;
        ">

            <span>
                {axis_html}
            </span>

            <span style="color:#777;">
                OVITO
            </span>
        </div>
        """
    )

    return VBox(
        [header, viewer, footer],
        layout=Layout(
            width="100%",
            border="1px solid #444",
            overflow="hidden",
        ),
    )


# ---------------------------------------------------------
# Legende cartesiane
# ---------------------------------------------------------

top_axes = """
<span style="color:#ff5555;">x →</span>
&nbsp;&nbsp;
<span style="color:#55cc55;">y ↑</span>
&nbsp;&nbsp;
<span style="color:#888;">view −z</span>
"""

front_axes = """
<span style="color:#ff5555;">x →</span>
&nbsp;&nbsp;
<span style="color:#668cff;">z ↑</span>
&nbsp;&nbsp;
<span style="color:#888;">view −y</span>
"""

left_axes = """
<span style="color:#55cc55;">y →</span>
&nbsp;&nbsp;
<span style="color:#668cff;">z ↑</span>
&nbsp;&nbsp;
<span style="color:#888;">view +x</span>
"""

perspective_axes = """
<span style="color:#ff5555;">x</span>
&nbsp;
<span style="color:#55cc55;">y</span>
&nbsp;
<span style="color:#668cff;">z</span>
&nbsp;&nbsp;
<span style="color:#888;">3D</span>
"""


# ---------------------------------------------------------
# Quattro viewport
# ---------------------------------------------------------

panels = [
    make_panel(
        Viewport.Type.Top,
        "Top",
        "XY",
        top_axes,
    ),
    make_panel(
        Viewport.Type.Front,
        "Front",
        "XZ",
        front_axes,
    ),
    make_panel(
        Viewport.Type.Left,
        "Left",
        "YZ",
        left_axes,
    ),
    make_panel(
        Viewport.Type.Perspective,
        "Perspective",
        "XYZ",
        perspective_axes,
        camera_dir=(1.0, 1.0, -0.8),
    ),
]


# ---------------------------------------------------------
# Layout finale 2 × 2
# ---------------------------------------------------------

grid = GridBox(
    panels,
    layout=Layout(
        width="100%",
        grid_template_columns="repeat(2, minmax(0, 1fr))",
        grid_gap="8px",
    ),
)

display(grid)

GridBox(children=(VBox(children=(HTML(value='\n        <div style="\n            background:#202225;\n        …

---


## Coordinazione locale e identificazione dell'ambiente atomico

La distanza tra l'atomo di riferimento $i$ e ogni altro atomo $j$ è

$$
r_{ij}
=
\sqrt{
(x_j-x_i)^2+
(y_j-y_i)^2+
(z_j-z_i)^2
}.
$$

Per un cristallo FCC con parametro reticolare $a$, la distanza tra primi vicini è

$$
d_{\mathrm{nn}}=\frac{a}{\sqrt{2}}.
$$

Nel caso dell'Ag considerato qui, con

$$
a \simeq 4.1604\ \text{Å},
$$

si ottiene

$$
d_{\mathrm{nn}}
\simeq
\frac{4.1604}{\sqrt{2}}
\simeq
2.94\ \text{Å}.
$$

Selezionando quindi gli atomi appartenenti alla prima shell di coordinazione,
possiamo calcolare il **coordination number**

$$
CN_i =
\sum_{j\neq i}
\Theta(r_c-r_{ij}),
$$

dove $r_c$ è un cutoff scelto in modo da includere la prima shell di vicini.

Per una struttura FCC ideale ci aspettiamo:

| Ambiente locale | Numero di primi vicini |
|---|---:|
| singolo piano FCC(100) | 4 |
| superficie FCC(100) | 8 |
| bulk FCC | 12 |

Un numero di coordinazione inferiore può inoltre indicare bordi, spigoli,
vacanze o altre forme di sottocoordinazione.

Il valore medio delle distanze dei primi vicini permette anche di stimare il
parametro reticolare FCC:

$$
a_{\mathrm{est}}
=
\sqrt{2}\,\langle d_{\mathrm{nn}}\rangle.
$$

> **Importante:** questa classificazione è una diagnosi locale basata sulla
> coordinazione. Non sostituisce metodi strutturali più rigorosi come CNA,
> PTM o l'analisi della radial distribution function $g(r)$.
>
> Inoltre, se il sistema è periodico, le distanze devono essere calcolate
> usando la minimum-image convention.

## DISTANZA TRA I NEAREST NEIGHBORS (NN) di un certo atomo 5 di riferimento
atomo 5 atomo di riferimento 

In [ ]:
import numpy as np
import ovito

from IPython.display import display
from ipywidgets import GridBox, Layout, VBox, HTML

from ovito.gui import create_ipywidget
from ovito.io import import_file
from ovito.modifiers import PythonScriptModifier
from ovito.vis import Viewport


# =========================================================
# 1. Parametri
# =========================================================

reference_atom = 5
reference_index = reference_atom - 1

# Intervallo di distanza da evidenziare
r_min = 2.0
r_max = 3.0


# =========================================================
# 2. Distanze dall'atomo di riferimento
# =========================================================

reference_position = coords[reference_index]

dr = coords - reference_position

distances = np.linalg.norm(
    dr,
    axis=1,
)


# =========================================================
# 3. Prima shell di coordinazione
# =========================================================

neighbor_mask = (distances > r_min) & (distances < r_max)

neighbor_indices = np.where(neighbor_mask)[0]

neighbor_distances = distances[neighbor_mask]

if len(neighbor_distances) == 0:
    raise ValueError(f"Nessun vicino trovato nell'intervallo {r_min} < d < {r_max} Å.")


# =========================================================
# 4. Raggio geometrico ricavato dai dati
# =========================================================
#
# Per FCC a sfere a contatto:
#
#     d_nn = 2 R
#
# quindi:
#
#     R = d_nn / 2
#

d_nn = np.mean(neighbor_distances)

atomic_radius = d_nn / 3.5

# Parametro reticolare FCC stimato:
a_est = np.sqrt(2.0) * d_nn


# =========================================================
# 5. Report
# =========================================================

print("=" * 62)
print(" ANALISI GEOMETRICA FCC")
print("=" * 62)

print(f"\nAtomo di riferimento : {reference_atom}")

print(
    "Posizione [Å]        : "
    f"({reference_position[0]:.4f}, "
    f"{reference_position[1]:.4f}, "
    f"{reference_position[2]:.4f})"
)

print(f"\nIntervallo selezione : {r_min:.2f} < d < {r_max:.2f} Å")

print(f"Numero di vicini     : {len(neighbor_indices)}")

print(f"d_nn medio           : {d_nn:.4f} Å")

print(f"R geometrico Ag      : {atomic_radius:.4f} Å")

print(f"a FCC stimato        : {a_est:.4f} Å")

print("\nAtomi evidenziati:")

for i in neighbor_indices:
    print(f"  Atom {i + 1:4d}   d = {distances[i]:.4f} Å")


# =========================================================
# 6. Pipeline OVITO
# =========================================================

highlight_pipeline = import_file(str(xyz_path))


# =========================================================
# 7. Colore + raggio fisico/geometrico
# =========================================================


def highlight_neighbors(frame, data):

    n = data.particles.count

    if n != len(coords):
        raise ValueError(
            f"OVITO contiene {n} particelle, "
            f"ma coords contiene {len(coords)} coordinate."
        )

    # -----------------------------------------------------
    # Tutti gli atomi hanno LO STESSO raggio geometrico
    # -----------------------------------------------------

    radii = np.full(
        n,
        atomic_radius,
        dtype=np.float32,
    )

    # -----------------------------------------------------
    # Default: Ag grigio
    # -----------------------------------------------------

    colors = np.full(
        (n, 3),
        [0.42, 0.42, 0.42],
        dtype=np.float32,
    )

    # -----------------------------------------------------
    # Prima shell: rosso
    # -----------------------------------------------------

    colors[neighbor_mask] = [
        1.00,
        0.08,
        0.08,
    ]

    # -----------------------------------------------------
    # Atomo di riferimento: giallo
    # -----------------------------------------------------

    colors[reference_index] = [
        1.00,
        0.72,
        0.00,
    ]

    # -----------------------------------------------------
    # Proprietà OVITO
    # -----------------------------------------------------

    data.particles_.create_property(
        "Color",
        data=colors,
    )

    data.particles_.create_property(
        "Radius",
        data=radii,
    )


highlight_pipeline.modifiers.append(PythonScriptModifier(function=highlight_neighbors))

highlight_pipeline.compute()


# =========================================================
# 8. Pulizia scena OVITO
# =========================================================

for old_pipeline in list(ovito.scene.pipelines):
    old_pipeline.remove_from_scene()

highlight_pipeline.add_to_scene()


# =========================================================
# 9. Pannello grafico
# =========================================================


def make_panel(
    view_type,
    title,
    plane,
    axis_html,
    camera_dir=None,
):

    kwargs = {"type": view_type}

    if camera_dir is not None:
        kwargs["camera_dir"] = camera_dir

    vp = Viewport(**kwargs)

    vp.zoom_all()

    viewer = create_ipywidget(
        vp,
        antialiasing=True,
        picking=False,
        layout=Layout(
            width="100%",
            height="310px",
        ),
    )

    header = HTML(
        value=f"""
        <div style="
            background:#202225;
            color:#f2f2f2;
            padding:7px 10px;
            font-family:-apple-system,
                        BlinkMacSystemFont,
                        'Segoe UI',
                        sans-serif;
            font-size:14px;
            border-bottom:1px solid #444;
            display:flex;
            align-items:center;
            justify-content:space-between;
        ">

            <span style="font-weight:600;">
                {title}
            </span>

            <span style="
                color:#aaa;
                font-size:12px;
            ">
                {plane}
            </span>

        </div>
        """
    )

    footer = HTML(
        value=f"""
        <div style="
            background:#202225;
            color:#ddd;
            padding:6px 10px;
            font-family:Menlo, Monaco, monospace;
            font-size:12px;
            border-top:1px solid #444;
            display:flex;
            align-items:center;
            justify-content:space-between;
            gap:10px;
        ">

            <span>
                {axis_html}
            </span>

            <span style="white-space:nowrap;">

                <span style="color:#ff1414;">
                    ●
                </span>
                {r_min:.1f} &lt; d &lt; {r_max:.1f} Å

                &nbsp;&nbsp;

                <span style="color:#ffc000;">
                    ●
                </span>
                atom {reference_atom}

            </span>

        </div>
        """
    )

    return VBox(
        [
            header,
            viewer,
            footer,
        ],
        layout=Layout(
            width="100%",
            border="1px solid #444",
            overflow="hidden",
        ),
    )


# =========================================================
# 10. Assi cartesiani
# =========================================================

top_axes = """
<span style="color:#ff5555;">x →</span>
&nbsp;
<span style="color:#55cc55;">y ↑</span>
&nbsp;&nbsp;
<span style="color:#888;">view −z</span>
"""

front_axes = """
<span style="color:#ff5555;">x →</span>
&nbsp;
<span style="color:#668cff;">z ↑</span>
&nbsp;&nbsp;
<span style="color:#888;">view −y</span>
"""

left_axes = """
<span style="color:#55cc55;">y →</span>
&nbsp;
<span style="color:#668cff;">z ↑</span>
&nbsp;&nbsp;
<span style="color:#888;">view +x</span>
"""

perspective_axes = """
<span style="color:#ff5555;">x</span>
&nbsp;
<span style="color:#55cc55;">y</span>
&nbsp;
<span style="color:#668cff;">z</span>
&nbsp;&nbsp;
<span style="color:#888;">3D</span>
"""


# =========================================================
# 11. Quattro viewport
# =========================================================

panels = [
    make_panel(
        Viewport.Type.Top,
        "Top",
        "XY",
        top_axes,
    ),
    make_panel(
        Viewport.Type.Front,
        "Front",
        "XZ",
        front_axes,
    ),
    make_panel(
        Viewport.Type.Left,
        "Left",
        "YZ",
        left_axes,
    ),
    make_panel(
        Viewport.Type.Perspective,
        "Perspective",
        "XYZ",
        perspective_axes,
        camera_dir=(1.0, 1.0, -0.8),
    ),
]


# =========================================================
# 12. Layout 2 × 2
# =========================================================

grid = GridBox(
    panels,
    layout=Layout(
        width="100%",
        grid_template_columns=("repeat(2, minmax(0, 1fr))"),
        grid_gap="8px",
    ),
)

display(grid)

 ANALISI GEOMETRICA FCC

Atomo di riferimento : 5
Posizione [Å]        : (0.0000, 8.3208, 0.0000)

Intervallo selezione : 2.00 < d < 3.00 Å
Numero di vicini     : 5
d_nn medio           : 2.9418 Å
R geometrico Ag      : 0.8405 Å
a FCC stimato        : 4.1604 Å

Atomi evidenziati:
  Atom    4   d = 2.9418 Å
  Atom    6   d = 2.9418 Å
  Atom   21   d = 2.9418 Å
  Atom   22   d = 2.9418 Å
  Atom   26   d = 2.9418 Å


GridBox(children=(VBox(children=(HTML(value='\n        <div style="\n            background:#202225;\n        …

In [ ]:
# ATOMO 75 COME RIFERIMENTO

import numpy as np
import ovito

from IPython.display import display
from ipywidgets import GridBox, Layout, VBox, HTML

from ovito.gui import create_ipywidget
from ovito.io import import_file
from ovito.modifiers import PythonScriptModifier
from ovito.vis import Viewport


# =========================================================
# 1. Parametri
# =========================================================

reference_atom = 75
reference_index = reference_atom - 1

# Intervallo di distanza da evidenziare
r_min = 2.0
r_max = 3.0


# =========================================================
# 2. Distanze dall'atomo di riferimento
# =========================================================

reference_position = coords[reference_index]

dr = coords - reference_position

distances = np.linalg.norm(
    dr,
    axis=1,
)


# =========================================================
# 3. Prima shell di coordinazione
# =========================================================

neighbor_mask = (distances > r_min) & (distances < r_max)

neighbor_indices = np.where(neighbor_mask)[0]

neighbor_distances = distances[neighbor_mask]

if len(neighbor_distances) == 0:
    raise ValueError(f"Nessun vicino trovato nell'intervallo {r_min} < d < {r_max} Å.")


# =========================================================
# 4. Raggio geometrico ricavato dai dati
# =========================================================
#
# Per FCC a sfere a contatto:
#
#     d_nn = 2 R
#
# quindi:
#
#     R = d_nn / 2
#

d_nn = np.mean(neighbor_distances)

atomic_radius = d_nn / 3.5

# Parametro reticolare FCC stimato:
a_est = np.sqrt(2.0) * d_nn


# =========================================================
# 5. Report
# =========================================================

print("=" * 62)
print(" ANALISI GEOMETRICA FCC")
print("=" * 62)

print(f"\nAtomo di riferimento : {reference_atom}")

print(
    "Posizione [Å]        : "
    f"({reference_position[0]:.4f}, "
    f"{reference_position[1]:.4f}, "
    f"{reference_position[2]:.4f})"
)

print(f"\nIntervallo selezione : {r_min:.2f} < d < {r_max:.2f} Å")

print(f"Numero di vicini     : {len(neighbor_indices)}")

print(f"d_nn medio           : {d_nn:.4f} Å")

print(f"R geometrico Ag      : {atomic_radius:.4f} Å")

print(f"a FCC stimato        : {a_est:.4f} Å")

print("\nAtomi evidenziati:")

for i in neighbor_indices:
    print(f"  Atom {i + 1:4d}   d = {distances[i]:.4f} Å")


# =========================================================
# 6. Pipeline OVITO
# =========================================================

highlight_pipeline = import_file(str(xyz_path))


# =========================================================
# 7. Colore + raggio fisico/geometrico
# =========================================================


def highlight_neighbors(frame, data):

    n = data.particles.count

    if n != len(coords):
        raise ValueError(
            f"OVITO contiene {n} particelle, "
            f"ma coords contiene {len(coords)} coordinate."
        )

    # -----------------------------------------------------
    # Tutti gli atomi hanno LO STESSO raggio geometrico
    # -----------------------------------------------------

    radii = np.full(
        n,
        atomic_radius,
        dtype=np.float32,
    )

    # -----------------------------------------------------
    # Default: Ag grigio
    # -----------------------------------------------------

    colors = np.full(
        (n, 3),
        [0.42, 0.42, 0.42],
        dtype=np.float32,
    )

    # -----------------------------------------------------
    # Prima shell: rosso
    # -----------------------------------------------------

    colors[neighbor_mask] = [
        1.00,
        0.08,
        0.08,
    ]

    # -----------------------------------------------------
    # Atomo di riferimento: giallo
    # -----------------------------------------------------

    colors[reference_index] = [
        1.00,
        0.72,
        0.00,
    ]

    # -----------------------------------------------------
    # Proprietà OVITO
    # -----------------------------------------------------

    data.particles_.create_property(
        "Color",
        data=colors,
    )

    data.particles_.create_property(
        "Radius",
        data=radii,
    )


highlight_pipeline.modifiers.append(PythonScriptModifier(function=highlight_neighbors))

highlight_pipeline.compute()


# =========================================================
# 8. Pulizia scena OVITO
# =========================================================

for old_pipeline in list(ovito.scene.pipelines):
    old_pipeline.remove_from_scene()

highlight_pipeline.add_to_scene()


# =========================================================
# 9. Pannello grafico
# =========================================================


def make_panel(
    view_type,
    title,
    plane,
    axis_html,
    camera_dir=None,
):

    kwargs = {"type": view_type}

    if camera_dir is not None:
        kwargs["camera_dir"] = camera_dir

    vp = Viewport(**kwargs)

    vp.zoom_all()

    viewer = create_ipywidget(
        vp,
        antialiasing=True,
        picking=False,
        layout=Layout(
            width="100%",
            height="310px",
        ),
    )

    header = HTML(
        value=f"""
        <div style="
            background:#202225;
            color:#f2f2f2;
            padding:7px 10px;
            font-family:-apple-system,
                        BlinkMacSystemFont,
                        'Segoe UI',
                        sans-serif;
            font-size:14px;
            border-bottom:1px solid #444;
            display:flex;
            align-items:center;
            justify-content:space-between;
        ">

            <span style="font-weight:600;">
                {title}
            </span>

            <span style="
                color:#aaa;
                font-size:12px;
            ">
                {plane}
            </span>

        </div>
        """
    )

    footer = HTML(
        value=f"""
        <div style="
            background:#202225;
            color:#ddd;
            padding:6px 10px;
            font-family:Menlo, Monaco, monospace;
            font-size:12px;
            border-top:1px solid #444;
            display:flex;
            align-items:center;
            justify-content:space-between;
            gap:10px;
        ">

            <span>
                {axis_html}
            </span>

            <span style="white-space:nowrap;">

                <span style="color:#ff1414;">
                    ●
                </span>
                {r_min:.1f} &lt; d &lt; {r_max:.1f} Å

                &nbsp;&nbsp;

                <span style="color:#ffc000;">
                    ●
                </span>
                atom {reference_atom}

            </span>

        </div>
        """
    )

    return VBox(
        [
            header,
            viewer,
            footer,
        ],
        layout=Layout(
            width="100%",
            border="1px solid #444",
            overflow="hidden",
        ),
    )


# =========================================================
# 10. Assi cartesiani
# =========================================================

top_axes = """
<span style="color:#ff5555;">x →</span>
&nbsp;
<span style="color:#55cc55;">y ↑</span>
&nbsp;&nbsp;
<span style="color:#888;">view −z</span>
"""

front_axes = """
<span style="color:#ff5555;">x →</span>
&nbsp;
<span style="color:#668cff;">z ↑</span>
&nbsp;&nbsp;
<span style="color:#888;">view −y</span>
"""

left_axes = """
<span style="color:#55cc55;">y →</span>
&nbsp;
<span style="color:#668cff;">z ↑</span>
&nbsp;&nbsp;
<span style="color:#888;">view +x</span>
"""

perspective_axes = """
<span style="color:#ff5555;">x</span>
&nbsp;
<span style="color:#55cc55;">y</span>
&nbsp;
<span style="color:#668cff;">z</span>
&nbsp;&nbsp;
<span style="color:#888;">3D</span>
"""


# =========================================================
# 11. Quattro viewport
# =========================================================

panels = [
    make_panel(
        Viewport.Type.Top,
        "Top",
        "XY",
        top_axes,
    ),
    make_panel(
        Viewport.Type.Front,
        "Front",
        "XZ",
        front_axes,
    ),
    make_panel(
        Viewport.Type.Left,
        "Left",
        "YZ",
        left_axes,
    ),
    make_panel(
        Viewport.Type.Perspective,
        "Perspective",
        "XYZ",
        perspective_axes,
        camera_dir=(1.0, 1.0, -0.8),
    ),
]


# =========================================================
# 12. Layout 2 × 2
# =========================================================

grid = GridBox(
    panels,
    layout=Layout(
        width="100%",
        grid_template_columns=("repeat(2, minmax(0, 1fr))"),
        grid_gap="8px",
    ),
)

display(grid)

 ANALISI GEOMETRICA FCC

Atomo di riferimento : 75
Posizione [Å]        : (4.1604, 4.1604, 8.3208)

Intervallo selezione : 2.00 < d < 3.00 Å
Numero di vicini     : 12
d_nn medio           : 2.9418 Å
R geometrico Ag      : 0.8405 Å
a FCC stimato        : 4.1604 Å

Atomi evidenziati:
  Atom   57   d = 2.9418 Å
  Atom   67   d = 2.9418 Å
  Atom   68   d = 2.9418 Å
  Atom   70   d = 2.9418 Å
  Atom   76   d = 2.9418 Å
  Atom   78   d = 2.9418 Å
  Atom   83   d = 2.9418 Å
  Atom   88   d = 2.9418 Å
  Atom   92   d = 2.9418 Å
  Atom   97   d = 2.9418 Å
  Atom   98   d = 2.9418 Å
  Atom  103   d = 2.9418 Å


GridBox(children=(VBox(children=(HTML(value='\n        <div style="\n            background:#202225;\n        …

In [3]:
# File di input con colonne: x y z
txt_path = Path("/Users/francesco/Desktop/computational-course/data/raw/fcc100a108.txt")

if not txt_path.exists():
    raise FileNotFoundError(f"File non trovato: {txt_path}")

# Legge le coordinate
coords = np.loadtxt(txt_path)

print(coords.shape)
print(coords[0, 1])

print(coords[2, 1])

print("\n")

x = 11
for i in range(3):
    print(coords[x + 1, i])  # 12esima riga

x = 2  # y
print("\n")
for i in range(10):
    print(coords[i, x - 1])

print("\n")
# troviamo una distanza between 2 and 3 from atom 5

natoms = coords.shape[0]

for i in range(0, natoms):
    dx = coords[4, 0] - coords[i, 0]
    dy = coords[4, 1] - coords[i, 1]
    dz = coords[4, 2] - coords[i, 2]
    d2 = dx * dx + dy * dy + dz * dz
    ds = np.sqrt(d2)
    if ds > 2.0 and ds < 3.0:
        print(f"Atom {i + 1} is at distance {ds:.4f} from atom 5")

print("\n")
# ora troviamo una distanza tra 2 e 3 da un altro atomo, ad esempio l'atomo 76
for i in range(0, natoms):
    dx = coords[12, 0] - coords[i, 0]
    dy = coords[12, 1] - coords[i, 1]
    dz = coords[12, 2] - coords[i, 2]
    d2 = dx * dx + dy * dy + dz * dz
    ds = np.sqrt(d2)
    if ds > 2.0 and ds < 3.0:
        print(f"Atom {i + 1} is at distance {ds:.4f} from atom 5")

(108, 3)
2.0802
4.1604


8.3208
0.0
0.0


2.0802
0.0
4.1604
6.2406
8.3208
10.401
0.0
2.0802
4.1604
6.2406


Atom 4 is at distance 2.9418 from atom 5
Atom 6 is at distance 2.9418 from atom 5
Atom 21 is at distance 2.9418 from atom 5
Atom 22 is at distance 2.9418 from atom 5
Atom 26 is at distance 2.9418 from atom 5


Atom 8 is at distance 2.9418 from atom 5
Atom 14 is at distance 2.9418 from atom 5
Atom 27 is at distance 2.9418 from atom 5
Atom 28 is at distance 2.9418 from atom 5
Atom 29 is at distance 2.9418 from atom 5


In [ ]:
from pathlib import Path

import numpy as np
import ovito

from IPython.display import display
from ipywidgets import GridBox, Layout, VBox, HTML

from ovito.gui import create_ipywidget
from ovito.io import import_file
from ovito.modifiers import PythonScriptModifier
from ovito.vis import Viewport

from pathlib import Path


# =========================================================
# Trova automaticamente la root del progetto
# =========================================================


def find_project_root(start=Path.cwd()):

    for folder in [start, *start.parents]:
        if (folder / "pyproject.toml").exists():
            return folder

    raise FileNotFoundError(
        "Non riesco a trovare la root del progetto (cerco pyproject.toml)."
    )


PROJECT_ROOT = find_project_root()

print("Project root:")
print(PROJECT_ROOT)


# =========================================================
# File da caricare
# =========================================================

filename = "fcc100a864.txt"

txt_path = PROJECT_ROOT / "data" / "raw" / filename


# =========================================================
# Controllo
# =========================================================

if not txt_path.exists():
    data_dir = PROJECT_ROOT / "data" / "raw"

    available = sorted(p.name for p in data_dir.glob("*.txt"))

    raise FileNotFoundError(
        f"\nFile non trovato:\n{txt_path}\n\n"
        f"File .txt disponibili in data/raw:\n" + "\n".join(available)
    )


print("\nFile trovato:")
print(txt_path)
# Consideriamo solo x, y, z
coords = np.loadtxt(txt_path)[:, :3]

natoms = len(coords)

print("=" * 65)
print(" STRUTTURA CARICATA")
print("=" * 65)

print(f"\nFile       : {txt_path.name}")
print(f"Atomi      : {natoms}")
print(f"Shape      : {coords.shape}")

print("\nPrime 5 coordinate:")
print(coords[:5])


# =========================================================
# 2. CONVERSIONE TXT -> XYZ PER OVITO
# =========================================================

OUT = Path("generated")
OUT.mkdir(exist_ok=True)

xyz_path = OUT / f"{txt_path.stem}.xyz"

with xyz_path.open(
    "w",
    encoding="utf-8",
) as f:
    f.write(f"{natoms}\n")

    f.write(f"Ag structure imported from {txt_path.name}\n")

    for x, y, z in coords:
        f.write(f"Ag {x:.8f} {y:.8f} {z:.8f}\n")


print("\nXYZ OVITO:")
print(xyz_path.resolve())


# =========================================================
# 3. PARAMETRI DELL'ANALISI
# =========================================================

# Numerazione "umana":
# atom 1 -> index 0
# atom 75 -> index 74

reference_atom = 200

if not 1 <= reference_atom <= natoms:
    raise ValueError(
        f"reference_atom={reference_atom}, ma il file contiene {natoms} atomi."
    )

reference_index = reference_atom - 1


# Finestra della prima shell per Ag FCC ideale
r_min = 2.0
r_max = 3.0


# =========================================================
# 4. DISTANZE DALL'ATOMO DI RIFERIMENTO
# =========================================================

reference_position = coords[reference_index]

# r_i - r_ref
dr = coords - reference_position

# Distanza euclidea
distances = np.linalg.norm(
    dr,
    axis=1,
)


# =========================================================
# 5. PRIMA SHELL DI COORDINAZIONE
# =========================================================

neighbor_mask = (distances > r_min) & (distances < r_max)

neighbor_indices = np.where(neighbor_mask)[0]

neighbor_distances = distances[neighbor_mask]

if len(neighbor_distances) == 0:
    raise ValueError(f"Nessun vicino trovato nell'intervallo {r_min} < d < {r_max} Å.")


# =========================================================
# 6. DISTANZA DI PRIMO VICINO E RAGGIO GEOMETRICO
# =========================================================
#
# Per FCC nel modello hard-sphere:
#
#     d_nn = 2 R
#
# quindi:
#
#     R = d_nn / 2
#

# Mediana: robusta anche con piccole distorsioni
d_nn = np.median(neighbor_distances)

atomic_radius = d_nn / 3.5

# Per FCC:
#
# d_nn = a / sqrt(2)
#
# quindi:
#
# a = sqrt(2) d_nn

a_est = np.sqrt(2.0) * d_nn


# =========================================================
# 7. CLASSIFICAZIONE LOCALE
# =========================================================

coordination_number = len(neighbor_indices)

if coordination_number == 12:
    environment = "FCC bulk"

elif coordination_number == 8:
    environment = "superficie FCC(100)"

elif coordination_number == 4:
    environment = "singolo layer FCC(100)"

elif coordination_number < 4:
    environment = "fortemente sottocoordinato / bordo"

elif coordination_number < 8:
    environment = "sottocoordinato / bordo / superficie incompleta"

elif coordination_number < 12:
    environment = "sottocoordinato rispetto al bulk FCC"

else:
    environment = "coordinazione > 12: controllare cutoff o struttura"


# =========================================================
# 8. REPORT NUMERICO
# =========================================================

print("\n" + "=" * 65)
print(" ANALISI GEOMETRICA LOCALE")
print("=" * 65)

print(f"\nAtomo di riferimento : {reference_atom}")

print(
    "Posizione [Å]        : "
    f"({reference_position[0]:.4f}, "
    f"{reference_position[1]:.4f}, "
    f"{reference_position[2]:.4f})"
)

print(f"Intervallo           : {r_min:.2f} < d < {r_max:.2f} Å")

print(f"Numero di vicini     : {coordination_number}")

print(f"Ambiente locale      : {environment}")

print(f"d_nn                 : {d_nn:.4f} Å")

print(f"R geometrico         : {atomic_radius:.4f} Å")

print(f"a FCC stimato        : {a_est:.4f} Å")


print("\nAtomi nella prima shell:")

for i in neighbor_indices:
    print(f"  Atom {i + 1:5d}    d = {distances[i]:.4f} Å")


# =========================================================
# 9. PIPELINE OVITO
# =========================================================

highlight_pipeline = import_file(str(xyz_path))


# =========================================================
# 10. COLORI + RAGGIO GEOMETRICO
# =========================================================


def highlight_neighbors(
    frame,
    data,
):

    n = data.particles.count

    if n != natoms:
        raise ValueError(
            f"OVITO contiene {n} particelle, ma il file ne contiene {natoms}."
        )

    # -----------------------------------------------------
    # Tutti gli atomi hanno LO STESSO raggio
    # -----------------------------------------------------

    radii = np.full(
        n,
        atomic_radius,
        dtype=np.float32,
    )

    # -----------------------------------------------------
    # Default: Ag grigio
    # -----------------------------------------------------

    colors = np.full(
        (n, 3),
        [0.42, 1.32, 0.42],
        dtype=np.float32,
    )

    # -----------------------------------------------------
    # Prima shell: rosso
    # -----------------------------------------------------

    colors[neighbor_mask] = [
        1.00,
        0.08,
        0.08,
    ]

    # -----------------------------------------------------
    # Atomo di riferimento: giallo
    # -----------------------------------------------------

    colors[reference_index] = [
        1.00,
        0.72,
        0.00,
    ]

    # -----------------------------------------------------
    # Proprietà OVITO
    # -----------------------------------------------------

    data.particles_.create_property(
        "Color",
        data=colors,
    )

    data.particles_.create_property(
        "Radius",
        data=radii,
    )


highlight_pipeline.modifiers.append(PythonScriptModifier(function=highlight_neighbors))

# Prima valutazione
highlight_pipeline.compute()


# =========================================================
# 11. PULIZIA SCENA OVITO
# =========================================================

for old_pipeline in list(ovito.scene.pipelines):
    old_pipeline.remove_from_scene()


highlight_pipeline.add_to_scene()


# =========================================================
# 12. FUNZIONE GRAFICA DEI PANNELLI
# =========================================================


def make_panel(
    view_type,
    title,
    plane,
    axis_html,
    camera_dir=None,
):

    kwargs = {"type": view_type}

    if camera_dir is not None:
        kwargs["camera_dir"] = camera_dir

    vp = Viewport(**kwargs)

    vp.zoom_all()

    viewer = create_ipywidget(
        vp,
        antialiasing=True,
        picking=False,
        layout=Layout(
            width="100%",
            height="310px",
        ),
    )

    # -----------------------------------------------------
    # Header
    # -----------------------------------------------------

    header = HTML(
        value=f"""
        <div style="
            background:#202225;
            color:#f2f2f2;
            padding:7px 10px;

            font-family:
                -apple-system,
                BlinkMacSystemFont,
                'Segoe UI',
                sans-serif;

            font-size:14px;

            border-bottom:
                1px solid #444;

            display:flex;
            align-items:center;
            justify-content:
                space-between;
        ">

            <span style="
                font-weight:600;
            ">
                {title}
            </span>

            <span style="
                color:#aaa;
                font-size:12px;
            ">
                {plane}
            </span>

        </div>
        """
    )

    # -----------------------------------------------------
    # Footer
    # -----------------------------------------------------

    footer = HTML(
        value=f"""
        <div style="
            background:#202225;
            color:#ddd;

            padding:
                6px 10px;

            font-family:
                Menlo,
                Monaco,
                monospace;

            font-size:12px;

            border-top:
                1px solid #444;

            display:flex;
            align-items:center;

            justify-content:
                space-between;

            gap:10px;
        ">

            <span>
                {axis_html}
            </span>

            <span style="
                white-space:nowrap;
            ">

                <span style="
                    color:#ff1414;
                ">
                    ●
                </span>

                {r_min:.1f}
                &lt; d &lt;
                {r_max:.1f} Å

                &nbsp;&nbsp;

                <span style="
                    color:#ffc000;
                ">
                    ●
                </span>

                atom {reference_atom}

            </span>

        </div>
        """
    )

    return VBox(
        [
            header,
            viewer,
            footer,
        ],
        layout=Layout(
            width="100%",
            border="1px solid #444",
            overflow="hidden",
        ),
    )


# =========================================================
# 13. ASSI CARTESIANI
# =========================================================

top_axes = """
<span style="
    color:#ff5555;
">
    x →
</span>

&nbsp;

<span style="
    color:#55cc55;
">
    y ↑
</span>

&nbsp;&nbsp;

<span style="
    color:#888;
">
    view −z
</span>
"""


front_axes = """
<span style="
    color:#ff5555;
">
    x →
</span>

&nbsp;

<span style="
    color:#668cff;
">
    z ↑
</span>

&nbsp;&nbsp;

<span style="
    color:#888;
">
    view −y
</span>
"""


left_axes = """
<span style="
    color:#55cc55;
">
    y →
</span>

&nbsp;

<span style="
    color:#668cff;
">
    z ↑
</span>

&nbsp;&nbsp;

<span style="
    color:#888;
">
    view +x
</span>
"""


perspective_axes = """
<span style="
    color:#ff5555;
">
    x
</span>

&nbsp;

<span style="
    color:#55cc55;
">
    y
</span>

&nbsp;

<span style="
    color:#668cff;
">
    z
</span>

&nbsp;&nbsp;

<span style="
    color:#888;
">
    3D
</span>
"""


# =========================================================
# 14. QUATTRO VIEWPORT
# =========================================================

panels = [
    make_panel(
        Viewport.Type.Top,
        "Top",
        "XY",
        top_axes,
    ),
    make_panel(
        Viewport.Type.Front,
        "Front",
        "XZ",
        front_axes,
    ),
    make_panel(
        Viewport.Type.Left,
        "Left",
        "YZ",
        left_axes,
    ),
    make_panel(
        Viewport.Type.Perspective,
        "Perspective",
        "XYZ",
        perspective_axes,
        camera_dir=(
            1.0,
            1.0,
            -0.8,
        ),
    ),
]


# =========================================================
# 15. LAYOUT FINALE 2 × 2
# =========================================================

grid = GridBox(
    panels,
    layout=Layout(
        width="100%",
        grid_template_columns=("repeat(2, minmax(0, 1fr))"),
        grid_gap="8px",
    ),
)

display(grid)

Project root:
/Users/francesco/Desktop/computational-course

File trovato:
/Users/francesco/Desktop/computational-course/data/raw/fcc100a864.txt
 STRUTTURA CARICATA

File       : fcc100a864.txt
Atomi      : 864
Shape      : (864, 3)

Prime 5 coordinate:
[[2.0802 2.0802 0.    ]
 [0.     0.     0.    ]
 [0.     4.1604 0.    ]
 [2.0802 6.2406 0.    ]
 [0.     8.3208 0.    ]]

XYZ OVITO:
/Users/francesco/Desktop/computational-course/book/generated/fcc100a864.xyz

 ANALISI GEOMETRICA LOCALE

Atomo di riferimento : 200
Posizione [Å]        : (10.4010, 10.4010, 4.1604)
Intervallo           : 2.00 < d < 3.00 Å
Numero di vicini     : 12
Ambiente locale      : FCC bulk
d_nn                 : 2.9418 Å
R geometrico         : 0.8405 Å
a FCC stimato        : 4.1604 Å

Atomi nella prima shell:
  Atom    84    d = 2.9418 Å
  Atom   117    d = 2.9418 Å
  Atom   118    d = 2.9418 Å
  Atom   125    d = 2.9418 Å
  Atom   187    d = 2.9418 Å
  Atom   195    d = 2.9418 Å
  Atom   199    d = 2.9418 Å
  Atom 

GridBox(children=(VBox(children=(HTML(value='\n        <div style="\n            background:#202225;\n        …

In [ ]:
from pathlib import Path

import numpy as np
import ovito

from IPython.display import display
from ipywidgets import GridBox, Layout, VBox, HTML

from ovito.gui import create_ipywidget
from ovito.io import import_file
from ovito.modifiers import PythonScriptModifier
from ovito.vis import Viewport


# =========================================================
# 0. CONFIGURAZIONE
# =========================================================

filename = "fcc100a864.txt"

# Atomo da analizzare, numerazione umana: 1, 2, 3, ...
reference_atom = 200

# Intervallo della prima shell
r_min = 2.0
r_max = 3.0

# Aspetto grafico degli atomi NON selezionati.
#
# Nel widget Jupyter la vera trasparenza non viene resa
# correttamente sul nostro setup, quindi riduciamo anche
# il raggio VISIVO degli atomi di contesto.
background_radius_scale = 0.38

# Questa proprietà viene comunque scritta nei dati OVITO
# ed è utile in OVITO Desktop / renderer che la supportano.
background_transparency = 0.85


# =========================================================
# 1. TROVA LA ROOT DEL PROGETTO
# =========================================================


def find_project_root(start=Path.cwd()):

    for folder in [start, *start.parents]:
        if (folder / "pyproject.toml").exists():
            return folder

    raise FileNotFoundError(
        "Non riesco a trovare la root del progetto (cerco pyproject.toml)."
    )


PROJECT_ROOT = find_project_root()

txt_path = PROJECT_ROOT / "data" / "raw" / filename

if not txt_path.exists():
    available = sorted(p.name for p in (PROJECT_ROOT / "data" / "raw").glob("*.txt"))

    raise FileNotFoundError(
        f"File non trovato:\n{txt_path}\n\n"
        "File .txt disponibili:\n" + "\n".join(available)
    )


print("Project root :", PROJECT_ROOT)
print("File         :", txt_path)


# =========================================================
# 2. LETTURA COORDINATE
# =========================================================

coords = np.loadtxt(txt_path)

if coords.ndim == 1:
    coords = coords.reshape(1, -1)

if coords.shape[1] < 3:
    raise ValueError("Il file deve contenere almeno tre colonne: x y z.")

coords = coords[:, :3]

natoms = len(coords)

print("Numero atomi :", natoms)
print("Shape        :", coords.shape)


# =========================================================
# 3. CONVERSIONE TXT -> XYZ
# =========================================================

OUT = Path("generated")
OUT.mkdir(exist_ok=True)

xyz_path = OUT / f"{txt_path.stem}.xyz"

with xyz_path.open(
    "w",
    encoding="utf-8",
) as f:
    f.write(f"{natoms}\n")

    f.write(f"Ag structure imported from {txt_path.name}\n")

    for x, y, z in coords:
        f.write(f"Ag {x:.8f} {y:.8f} {z:.8f}\n")


print("XYZ OVITO    :", xyz_path.resolve())


# =========================================================
# 4. ATOMO DI RIFERIMENTO
# =========================================================

if not 1 <= reference_atom <= natoms:
    raise ValueError(
        f"reference_atom={reference_atom}, ma il file contiene {natoms} atomi."
    )

reference_index = reference_atom - 1

reference_position = coords[reference_index]


# =========================================================
# 5. DISTANZE
# =========================================================
#
# Distanza euclidea ordinaria:
#
#   d_i = |r_i - r_ref|
#
# Nota:
# qui NON stiamo ancora applicando PBC/minimum image.
# =========================================================

dr = coords - reference_position

distances = np.linalg.norm(
    dr,
    axis=1,
)


# =========================================================
# 6. PRIMA SHELL
# =========================================================

neighbor_mask = (distances > r_min) & (distances < r_max)

neighbor_indices = np.where(neighbor_mask)[0]

neighbor_distances = distances[neighbor_mask]

coordination_number = len(neighbor_indices)

if coordination_number == 0:
    raise ValueError(f"Nessun vicino trovato nell'intervallo {r_min} < d < {r_max} Å.")


# =========================================================
# 7. DISTANZA NN + RAGGIO GEOMETRICO FCC
# =========================================================
#
# Modello hard-sphere FCC:
#
#       d_nn = 2 R
#
# quindi:
#
#       R = d_nn / 2
#
# =========================================================

d_nn = np.median(neighbor_distances)

atomic_radius = d_nn / 2.0

# FCC:
#
# d_nn = a / sqrt(2)
#
# quindi:
#
# a = sqrt(2) d_nn

a_est = np.sqrt(2.0) * d_nn


# =========================================================
# 8. CLASSIFICAZIONE LOCALE
# =========================================================

if coordination_number == 12:
    environment = "FCC bulk"

elif coordination_number == 8:
    environment = "superficie FCC(100)"

elif coordination_number == 4:
    environment = "singolo layer FCC(100)"

elif coordination_number < 4:
    environment = "fortemente sottocoordinato / bordo"

elif coordination_number < 8:
    environment = "sottocoordinato / bordo / superficie"

elif coordination_number < 12:
    environment = "sottocoordinato rispetto al bulk FCC"

else:
    environment = "coordinazione > 12: controllare cutoff"


# =========================================================
# 9. REPORT
# =========================================================

print("\n" + "=" * 65)
print(" ANALISI GEOMETRICA LOCALE")
print("=" * 65)

print(f"Atomo di riferimento : {reference_atom}")

print(
    "Posizione [Å]        : "
    f"({reference_position[0]:.4f}, "
    f"{reference_position[1]:.4f}, "
    f"{reference_position[2]:.4f})"
)

print(f"Intervallo           : {r_min:.2f} < d < {r_max:.2f} Å")

print(f"Numero di vicini     : {coordination_number}")

print(f"Ambiente locale      : {environment}")

print(f"d_nn                 : {d_nn:.4f} Å")

print(f"R geometrico FCC     : {atomic_radius:.4f} Å")

print(f"a FCC stimato        : {a_est:.4f} Å")


# Stampiamo al massimo 20 vicini
print("\nPrimi vicini:")

max_print = 20

for i in neighbor_indices[:max_print]:
    print(f"  Atom {i + 1:5d}    d = {distances[i]:.4f} Å")

if coordination_number > max_print:
    print(f"  ... altri {coordination_number - max_print} atomi non mostrati.")


# =========================================================
# 10. PIPELINE OVITO
# =========================================================

highlight_pipeline = import_file(str(xyz_path))


# =========================================================
# 11. COLORI + RAGGI + TRASPARENZA
# =========================================================


def highlight_neighbors(frame, data):

    n = data.particles.count

    if n != natoms:
        raise ValueError(
            f"OVITO contiene {n} particelle, ma il file ne contiene {natoms}."
        )

    # -----------------------------------------------------
    # RAGGI VISIVI
    # -----------------------------------------------------
    #
    # Background:
    # più piccolo per simulare un effetto "ghost"
    # nel widget Jupyter.
    #
    # Rossi e giallo:
    # raggio geometrico FCC completo.
    # -----------------------------------------------------

    radii = np.full(
        n,
        atomic_radius * background_radius_scale,
        dtype=np.float32,
    )

    radii[neighbor_mask] = atomic_radius

    radii[reference_index] = atomic_radius

    # -----------------------------------------------------
    # COLORI
    # -----------------------------------------------------

    # Background chiaro
    colors = np.full(
        (n, 3),
        [0.72, 0.72, 0.72],
        dtype=np.float32,
    )

    # Prima shell: rosso
    colors[neighbor_mask] = [
        1.00,
        0.05,
        0.05,
    ]

    # Riferimento: giallo
    colors[reference_index] = [
        1.00,
        0.72,
        0.00,
    ]

    # -----------------------------------------------------
    # TRASPARENZA
    # -----------------------------------------------------
    #
    # 0 = opaco
    # 1 = completamente trasparente
    #
    # Viene salvata nella pipeline OVITO.
    # Il widget Jupyter può però non mostrarla
    # correttamente.
    # -----------------------------------------------------

    transparencies = np.full(
        n,
        background_transparency,
        dtype=np.float32,
    )

    # Rossi completamente opachi
    transparencies[neighbor_mask] = 0.0

    # Giallo completamente opaco
    transparencies[reference_index] = 0.0

    # -----------------------------------------------------
    # PROPRIETÀ OVITO
    # -----------------------------------------------------

    data.particles_.create_property(
        "Color",
        data=colors,
    )

    data.particles_.create_property(
        "Radius",
        data=radii,
    )

    data.particles_.create_property(
        "Transparency",
        data=transparencies,
    )


highlight_pipeline.modifiers.append(PythonScriptModifier(function=highlight_neighbors))

# Forza il calcolo
preview_data = highlight_pipeline.compute()


# =========================================================
# 12. CONTROLLO DELLE PROPRIETÀ
# =========================================================
#
# Questo serve a verificare che la trasparenza
# sia DAVVERO presente nei dati, anche se il widget
# non la rende.
# =========================================================

T = np.asarray(preview_data.particles["Transparency"])

R = np.asarray(preview_data.particles["Radius"])

print("\n" + "=" * 65)
print(" CONTROLLO OVITO")
print("=" * 65)

print(
    "Transparency background :",
    np.unique(
        np.round(
            T[~neighbor_mask & (np.arange(natoms) != reference_index)],
            3,
        )
    )[:5],
)

print(
    "Transparency vicini    :",
    np.unique(T[neighbor_mask]),
)

print(
    "Transparency reference :",
    T[reference_index],
)

print(f"R visuale background   : {atomic_radius * background_radius_scale:.4f} Å")

print(f"R reale evidenziati    : {atomic_radius:.4f} Å")


# =========================================================
# 13. PULIZIA SCENA
# =========================================================

for old_pipeline in list(ovito.scene.pipelines):
    old_pipeline.remove_from_scene()


highlight_pipeline.add_to_scene()


# =========================================================
# 14. PANNELLO GRAFICO
# =========================================================


def make_panel(
    view_type,
    title,
    plane,
    axis_html,
    camera_dir=None,
):

    kwargs = {"type": view_type}

    if camera_dir is not None:
        kwargs["camera_dir"] = camera_dir

    vp = Viewport(**kwargs)

    vp.zoom_all()

    viewer = create_ipywidget(
        vp,
        antialiasing=True,
        picking=False,
        layout=Layout(
            width="100%",
            height="310px",
        ),
    )

    header = HTML(
        value=f"""
        <div style="
            background:#202225;
            color:#f2f2f2;

            padding:7px 10px;

            font-family:
                -apple-system,
                BlinkMacSystemFont,
                'Segoe UI',
                sans-serif;

            font-size:14px;

            border-bottom:
                1px solid #444;

            display:flex;
            align-items:center;
            justify-content:space-between;
        ">

            <span style="
                font-weight:600;
            ">
                {title}
            </span>

            <span style="
                color:#aaa;
                font-size:12px;
            ">
                {plane}
            </span>

        </div>
        """
    )

    footer = HTML(
        value=f"""
        <div style="
            background:#202225;
            color:#ddd;

            padding:6px 10px;

            font-family:
                Menlo,
                Monaco,
                monospace;

            font-size:12px;

            border-top:
                1px solid #444;

            display:flex;
            align-items:center;
            justify-content:space-between;

            gap:10px;
        ">

            <span>
                {axis_html}
            </span>

            <span style="
                white-space:nowrap;
            ">

                <span style="
                    color:#ff1414;
                ">
                    ●
                </span>

                {r_min:.1f}
                &lt; d &lt;
                {r_max:.1f} Å

                &nbsp;&nbsp;

                <span style="
                    color:#ffc000;
                ">
                    ●
                </span>

                atom {reference_atom}

                &nbsp;&nbsp;

                <span style="
                    color:#aaa;
                ">
                    ● context
                </span>

            </span>

        </div>
        """
    )

    return VBox(
        [
            header,
            viewer,
            footer,
        ],
        layout=Layout(
            width="100%",
            border="1px solid #444",
            overflow="hidden",
        ),
    )


# =========================================================
# 15. ASSI
# =========================================================

top_axes = """
<span style="color:#ff5555;">
x →
</span>

&nbsp;

<span style="color:#55cc55;">
y ↑
</span>

&nbsp;&nbsp;

<span style="color:#888;">
view −z
</span>
"""


front_axes = """
<span style="color:#ff5555;">
x →
</span>

&nbsp;

<span style="color:#668cff;">
z ↑
</span>

&nbsp;&nbsp;

<span style="color:#888;">
view −y
</span>
"""


left_axes = """
<span style="color:#55cc55;">
y →
</span>

&nbsp;

<span style="color:#668cff;">
z ↑
</span>

&nbsp;&nbsp;

<span style="color:#888;">
view +x
</span>
"""


perspective_axes = """
<span style="color:#ff5555;">
x
</span>

&nbsp;

<span style="color:#55cc55;">
y
</span>

&nbsp;

<span style="color:#668cff;">
z
</span>

&nbsp;&nbsp;

<span style="color:#888;">
3D
</span>
"""


# =========================================================
# 16. QUATTRO VIEWPORT
# =========================================================

panels = [
    make_panel(
        Viewport.Type.Top,
        "Top",
        "XY",
        top_axes,
    ),
    make_panel(
        Viewport.Type.Front,
        "Front",
        "XZ",
        front_axes,
    ),
    make_panel(
        Viewport.Type.Left,
        "Left",
        "YZ",
        left_axes,
    ),
    make_panel(
        Viewport.Type.Perspective,
        "Perspective",
        "XYZ",
        perspective_axes,
        camera_dir=(
            1.0,
            1.0,
            -0.8,
        ),
    ),
]


# =========================================================
# 17. GRID 2 × 2
# =========================================================

grid = GridBox(
    panels,
    layout=Layout(
        width="100%",
        grid_template_columns=("repeat(2, minmax(0, 1fr))"),
        grid_gap="8px",
    ),
)

display(grid)

Project root : /Users/francesco/Desktop/computational-course
File         : /Users/francesco/Desktop/computational-course/data/raw/fcc100a864.txt
Numero atomi : 864
Shape        : (864, 3)
XYZ OVITO    : /Users/francesco/Desktop/computational-course/book/generated/fcc100a864.xyz

 ANALISI GEOMETRICA LOCALE
Atomo di riferimento : 200
Posizione [Å]        : (10.4010, 10.4010, 4.1604)
Intervallo           : 2.00 < d < 3.00 Å
Numero di vicini     : 12
Ambiente locale      : FCC bulk
d_nn                 : 2.9418 Å
R geometrico FCC     : 1.4709 Å
a FCC stimato        : 4.1604 Å

Primi vicini:
  Atom    84    d = 2.9418 Å
  Atom   117    d = 2.9418 Å
  Atom   118    d = 2.9418 Å
  Atom   125    d = 2.9418 Å
  Atom   187    d = 2.9418 Å
  Atom   195    d = 2.9418 Å
  Atom   199    d = 2.9418 Å
  Atom   211    d = 2.9418 Å
  Atom   222    d = 2.9418 Å
  Atom   225    d = 2.9418 Å
  Atom   277    d = 2.9418 Å
  Atom   278    d = 2.9418 Å

 CONTROLLO OVITO
Transparency background : [0.85]
Transp

GridBox(children=(VBox(children=(HTML(value='\n        <div style="\n            background:#202225;\n        …